# MSD Special State With Noiseless Prefix And Pipeline-Faithful Noisy Suffix

This notebook keeps the logical compilation pipeline for the noisy suffix. The workflow is:

1. define the `MSD` suffix in the Gemini logical dialect
2. compile it through `build_task(..., noisy_initializer=make_noisy_steane7_initializer(sim))`
3. inspect the emitted physical squin kernel and locate the five initializer invokes
4. replace only those five initializer invokes with one custom prefix call that does:
   - basis-dependent noiseless tomography-inverse prep on the measured input qubit
   - noiseless 5-qubit physical `MSD` inverse
   - the same noisy Steane initializer used by the pipeline

So the suffix stays compiler-emitted, and the only surgical edit is at the initializer boundary.


In [1]:
from IPython.display import HTML, display
from pathlib import Path
import sys

import numpy as np
from kirin.dialects import func, ilist

from bloqade import qubit, squin
from bloqade.gemini import logical as gemini_logical
from bloqade.gemini.logical.stdlib import default_post_processing
from bloqade.lanes import GeminiLogicalSimulator
from bloqade.tsim import Circuit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "demo").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from demo.msd_utils.circuits import build_task, make_noisy_steane7_initializer
from demo.msd_utils.core import run_task

BASIS_LABELS = ("X", "Y", "Z")


In [2]:
def show_diagram(diagram_html: str, *, height: int = 360) -> None:
    display(
        HTML(
            f'<div style="overflow-x:auto; width:100%; border:1px solid #ddd; padding:8px; background:white; max-height:{height}px;">'
            + diagram_html
            + "</div>"
        )
    )


## Logical Suffix And Noiseless Physical Inverse


In [3]:
@squin.kernel
def logical_msd_forward(reg):
    squin.broadcast.sqrt_x([reg[0], reg[1], reg[4]])
    squin.broadcast.cz([reg[0], reg[2]], [reg[1], reg[3]])
    squin.broadcast.sqrt_y([reg[0], reg[3]])
    squin.broadcast.cz([reg[0], reg[3]], [reg[2], reg[4]])
    squin.sqrt_x_adj(reg[0])
    squin.broadcast.cz([reg[0], reg[1]], [reg[4], reg[3]])
    squin.broadcast.sqrt_x_adj(reg)


@gemini_logical.kernel(aggressive_unroll=True)
def logical_suffix_x():
    reg = qubit.qalloc(5)
    logical_msd_forward(reg)
    squin.h(reg[0])
    return default_post_processing(reg)


@gemini_logical.kernel(aggressive_unroll=True)
def logical_suffix_y():
    reg = qubit.qalloc(5)
    logical_msd_forward(reg)
    squin.sqrt_z_adj(reg[0])
    squin.h(reg[0])
    return default_post_processing(reg)


@gemini_logical.kernel(aggressive_unroll=True)
def logical_suffix_z():
    reg = qubit.qalloc(5)
    logical_msd_forward(reg)
    return default_post_processing(reg)


@squin.kernel
def physical_msd_inverse(inputs):
    squin.broadcast.sqrt_x(inputs)
    squin.broadcast.cz([inputs[0], inputs[1]], [inputs[4], inputs[3]])
    squin.sqrt_x(inputs[0])
    squin.broadcast.cz([inputs[0], inputs[3]], [inputs[2], inputs[4]])
    squin.broadcast.sqrt_y_adj([inputs[0], inputs[3]])
    squin.broadcast.cz([inputs[0], inputs[2]], [inputs[1], inputs[3]])
    squin.broadcast.sqrt_x_adj([inputs[0], inputs[1], inputs[4]])


sim = GeminiLogicalSimulator()
logical_suffix_kernels = {
    "X": logical_suffix_x,
    "Y": logical_suffix_y,
    "Z": logical_suffix_z,
}
noisy_initializer = make_noisy_steane7_initializer(sim)


## Prefix Kernels

These kernels only replace the initialization boundary. Each one takes the same 35 physical qubits that the compiled physical squin kernel already allocated.


In [4]:
def build_prefix_kernels(noisy_init):
    @squin.kernel
    def prefix_x(q):
        q0 = q[0:7]
        q1 = q[7:14]
        q2 = q[14:21]
        q3 = q[21:28]
        q4 = q[28:35]
        inputs = [q0[6], q1[6], q2[6], q3[6], q4[6]]

        squin.h(q0[6])
        physical_msd_inverse(inputs)
        noisy_init(0.0, 0.0, 0.0, q0)
        noisy_init(0.0, 0.0, 0.0, q1)
        noisy_init(0.0, 0.0, 0.0, q2)
        noisy_init(0.0, 0.0, 0.0, q3)
        noisy_init(0.0, 0.0, 0.0, q4)

    @squin.kernel
    def prefix_y(q):
        q0 = q[0:7]
        q1 = q[7:14]
        q2 = q[14:21]
        q3 = q[21:28]
        q4 = q[28:35]
        inputs = [q0[6], q1[6], q2[6], q3[6], q4[6]]

        squin.h(q0[6])
        squin.sqrt_z(q0[6])
        physical_msd_inverse(inputs)
        noisy_init(0.0, 0.0, 0.0, q0)
        noisy_init(0.0, 0.0, 0.0, q1)
        noisy_init(0.0, 0.0, 0.0, q2)
        noisy_init(0.0, 0.0, 0.0, q3)
        noisy_init(0.0, 0.0, 0.0, q4)

    @squin.kernel
    def prefix_z(q):
        q0 = q[0:7]
        q1 = q[7:14]
        q2 = q[14:21]
        q3 = q[21:28]
        q4 = q[28:35]
        inputs = [q0[6], q1[6], q2[6], q3[6], q4[6]]

        physical_msd_inverse(inputs)
        noisy_init(0.0, 0.0, 0.0, q0)
        noisy_init(0.0, 0.0, 0.0, q1)
        noisy_init(0.0, 0.0, 0.0, q2)
        noisy_init(0.0, 0.0, 0.0, q3)
        noisy_init(0.0, 0.0, 0.0, q4)

    return {"X": prefix_x, "Y": prefix_y, "Z": prefix_z}


prefix_kernels = build_prefix_kernels(noisy_initializer)


## IR Surgery At The Initializer Boundary


In [5]:
def count_initializer_invokes(method, *, initializer_name: str = "noisy_steane7_initialize") -> int:
    block = method.code.body.blocks[0]
    count = 0
    for stmt in block.stmts:
        if isinstance(stmt, func.Invoke):
            callee = getattr(stmt, "callee", None)
            callee_name = getattr(callee, "sym_name", None) or getattr(callee, "name", None) or ""
            if initializer_name in str(callee_name):
                count += 1
    return count


def splice_noiseless_prefix_into_compiled_kernel(
    task,
    prefix_kernel,
    *,
    initializer_name: str = "noisy_steane7_initialize",
):
    mt = task.physical_squin_kernel.similar()
    block = mt.code.body.blocks[0]
    stmts = list(block.stmts)

    qubit_new_stmts = [stmt for stmt in stmts if "qubit.new" in stmt.name or stmt.name == "new"]
    if len(qubit_new_stmts) < 35:
        raise RuntimeError(f"Expected at least 35 qubit allocations, got {len(qubit_new_stmts)}")

    init_invokes = []
    for stmt in stmts:
        if isinstance(stmt, func.Invoke):
            callee = getattr(stmt, "callee", None)
            callee_name = getattr(callee, "sym_name", None) or getattr(callee, "name", None) or ""
            if initializer_name in str(callee_name):
                init_invokes.append(stmt)

    if len(init_invokes) != 5:
        raise RuntimeError(f"Expected 5 initializer invokes, got {len(init_invokes)}")

    anchor = init_invokes[0]
    full_reg = ilist.New(tuple(stmt.result for stmt in qubit_new_stmts[:35]))
    full_reg.insert_before(anchor)
    func.Invoke((full_reg.result,), callee=prefix_kernel).insert_before(anchor)

    for stmt in reversed(init_invokes):
        stmt.delete(safe=False)

    return mt


## Build The Compiler-Generated Tasks And Splice The Prefix In


In [6]:
compiled_tasks = {
    basis: build_task(
        sim,
        logical_suffix_kernels[basis],
        m2dets=None,
        m2obs=None,
        noisy_initializer=noisy_initializer,
        append_measurements=False,
    )
    for basis in BASIS_LABELS
}

compiled_initializer_counts = {
    basis: count_initializer_invokes(task.physical_squin_kernel)
    for basis, task in compiled_tasks.items()
}

pipeline_faithful_tasks = {}
for basis in BASIS_LABELS:
    task = build_task(
        sim,
        logical_suffix_kernels[basis],
        m2dets=None,
        m2obs=None,
        noisy_initializer=noisy_initializer,
        append_measurements=False,
    )
    task.__dict__["physical_squin_kernel"] = splice_noiseless_prefix_into_compiled_kernel(
        task,
        prefix_kernels[basis],
    )
    pipeline_faithful_tasks[basis] = task

spliced_initializer_counts = {
    basis: count_initializer_invokes(task.physical_squin_kernel)
    for basis, task in pipeline_faithful_tasks.items()
}

{
    "compiled_initializer_counts": compiled_initializer_counts,
    "spliced_initializer_counts": spliced_initializer_counts,
}


{'compiled_initializer_counts': {'X': 5, 'Y': 5, 'Z': 5},
 'spliced_initializer_counts': {'X': 0, 'Y': 0, 'Z': 0}}

In [7]:
# compiled_tasks["X"].physical_squin_kernel.print()

In [8]:
pipeline_faithful_tasks["X"].tsim_circuit.detector_error_model(approximate_disjoint_errors=True)

stim.DetectorErrorModel('''
    error(0.0108277) D0 D1 D2
    error(0.000285679) D0 D1 D2 D3 D4 D5
    error(0.00213829) D0 D1 D2 D3 D4 D5 D6 D7 D8
    error(0.00364461) D0 D1 D2 D3 D4 D5 D6 D7 D8 D9 D10 D11
    error(0.00014286) D0 D1 D2 D3 D4 D5 D6 D7 D8 D9 D10 D11 L2
    error(0.000780549) D0 D1 D2 D3 D4 D5 D6 D8 D9 D10 D11
    error(0.00014286) D0 D1 D2 D3 D4 D5 D6 D8 D9 D10 D11 L2
    error(0.000923251) D0 D1 D2 D3 D4 D5 D6 D8 D13
    error(0.00142814) D0 D1 D2 D3 D4 D5 D7 D9 D10 D11
    error(0.00014286) D0 D1 D2 D3 D4 D5 D7 D12 D14
    error(0.00172835) D0 D1 D2 D3 D4 D5 D9 D10 D11
    error(0.000285679) D0 D1 D2 D3 D4 D5 D9 D10 D11 D12 D13 D14
    error(0.00405198) D0 D1 D2 D3 D4 D5 D12 D13 D14
    error(0.00014286) D0 D1 D2 D3 D4 D5 D12 D13 D14 L0 L1 L2
    error(0.00233947) D0 D1 D2 D6 D7 D8
    error(0.00014286) D0 D1 D2 D6 D7 D8 D9 D10 D11
    error(0.00014286) D0 D1 D2 D6 D7 D8 D9 D10 D11 D12 D13 D14
    error(0.00370878) D0 D1 D2 D6 D7 D8 D12 D13 D14
    error(0.00342402)

## Verify Noiseless Determinism And Noisy Sampling


In [9]:
def summarize_task(task, *, shots: int = 256) -> dict[str, object]:
    noiseless = run_task(task, shots, with_noise=False)
    noisy = run_task(task, shots, with_noise=True)
    return {
        "noiseless_all_zero_detectors": bool(np.all(noiseless.detectors == 0)),
        "noiseless_unique_observables": np.unique(noiseless.observables, axis=0).tolist(),
        "dem_size": (task.detector_error_model.num_detectors, task.detector_error_model.num_observables),
        "noisy_sample_shapes": (tuple(noisy.detectors.shape), tuple(noisy.observables.shape)),
    }


PIPELINE_FAITHFUL_RESULTS = {
    basis: summarize_task(task)
    for basis, task in pipeline_faithful_tasks.items()
}

PIPELINE_FAITHFUL_RESULTS


{'X': {'noiseless_all_zero_detectors': True,
  'noiseless_unique_observables': [[0, 0, 0, 0, 0],
   [0, 0, 0, 0, 1],
   [0, 0, 0, 1, 0],
   [0, 0, 0, 1, 1],
   [0, 0, 1, 0, 0],
   [0, 0, 1, 0, 1],
   [0, 0, 1, 1, 0],
   [0, 0, 1, 1, 1],
   [0, 1, 0, 0, 0],
   [0, 1, 0, 0, 1],
   [0, 1, 0, 1, 0],
   [0, 1, 0, 1, 1],
   [0, 1, 1, 0, 0],
   [0, 1, 1, 0, 1],
   [0, 1, 1, 1, 0],
   [0, 1, 1, 1, 1],
   [1, 0, 0, 0, 0],
   [1, 0, 0, 0, 1],
   [1, 0, 0, 1, 0],
   [1, 0, 0, 1, 1],
   [1, 0, 1, 0, 0],
   [1, 0, 1, 0, 1],
   [1, 0, 1, 1, 0],
   [1, 0, 1, 1, 1],
   [1, 1, 0, 0, 0],
   [1, 1, 0, 0, 1],
   [1, 1, 0, 1, 0],
   [1, 1, 0, 1, 1],
   [1, 1, 1, 0, 0],
   [1, 1, 1, 0, 1],
   [1, 1, 1, 1, 0],
   [1, 1, 1, 1, 1]],
  'dem_size': (15, 5),
  'noisy_sample_shapes': ((256, 15), (256, 5))},
 'Y': {'noiseless_all_zero_detectors': True,
  'noiseless_unique_observables': [[0, 0, 0, 0, 0],
   [0, 0, 0, 0, 1],
   [0, 0, 0, 1, 0],
   [0, 0, 0, 1, 1],
   [0, 0, 1, 0, 0],
   [0, 0, 1, 0, 1],
   [0, 0, 1, 

## Derived Prefix From Logical Encoding Plus MSD Inverse

This construction matches the baseline you described:

1. compile a **noiseless** reference circuit for the logical encoding plus `MSD` unitary
2. trim off the terminal measurements / detector annotations
3. invert that unitary circuit
4. prepend it in front of the ordinary compiled noisy suffix circuit

So the prefix is the inverse of the compiled logical encoding-plus-`MSD` block, not the hand-written physical `MSD` inverse alone.


In [10]:
NONUNITARY_PREFIXES = (
    "M",
    "MX",
    "MY",
    "MR",
    "MRX",
    "MRY",
    "MPP",
    "DETECTOR",
    "OBSERVABLE_INCLUDE",
)


@gemini_logical.kernel(aggressive_unroll=True)
def logical_encode_plus_msd():
    reg = qubit.qalloc(5)
    logical_msd_forward(reg)
    return default_post_processing(reg)


def first_nonunitary_instruction_index(circuit: Circuit) -> int:
    for idx in range(len(circuit)):
        if str(circuit[idx]).startswith(NONUNITARY_PREFIXES):
            return idx
    return len(circuit)


logical_encode_plus_msd_task = build_task(
    sim,
    logical_encode_plus_msd,
    m2dets=None,
    m2obs=None,
    append_measurements=False,
)

logical_encode_plus_msd_unitary = logical_encode_plus_msd_task.tsim_circuit[: first_nonunitary_instruction_index(logical_encode_plus_msd_task.tsim_circuit)]
logical_encode_plus_msd_inverse_prefix = logical_encode_plus_msd_unitary.without_noise().inverse()

logical_inverse_prefix_tsim_circuits = {
    basis: logical_encode_plus_msd_inverse_prefix + compiled_tasks[basis].tsim_circuit
    for basis in BASIS_LABELS
}

LOGICAL_INVERSE_PREFIX_STATS = {
    basis: {
        "inverse_prefix_length": len(logical_encode_plus_msd_inverse_prefix),
        "compiled_suffix_length": len(compiled_tasks[basis].tsim_circuit),
        "derived_total_length": len(logical_inverse_prefix_tsim_circuits[basis]),
    }
    for basis in BASIS_LABELS
}

LOGICAL_INVERSE_PREFIX_STATS


{'X': {'inverse_prefix_length': 52,
  'compiled_suffix_length': 489,
  'derived_total_length': 541},
 'Y': {'inverse_prefix_length': 52,
  'compiled_suffix_length': 492,
  'derived_total_length': 544},
 'Z': {'inverse_prefix_length': 52,
  'compiled_suffix_length': 480,
  'derived_total_length': 532}}

## Visualize The Logical-Inverse-Prefix Tsim Circuit

The first diagram is the ordinary compiled noisy suffix. The second prepends the inverse of the compiled logical encoding-plus-`MSD` block.


In [11]:
VISUALIZE_BASIS = "X"

print(f"Ordinary compiled noisy suffix tsim circuit ({VISUALIZE_BASIS})")
show_diagram(str(compiled_tasks[VISUALIZE_BASIS].tsim_circuit.diagram(height=520)), height=5000)

print(f"Logical-inverse-prefix / compiled noisy suffix tsim circuit ({VISUALIZE_BASIS})")
show_diagram(str(logical_inverse_prefix_tsim_circuits[VISUALIZE_BASIS].diagram(height=520)), height=5000)


Ordinary compiled noisy suffix tsim circuit (X)


Logical-inverse-prefix / compiled noisy suffix tsim circuit (X)


## Save The Diagram As PNG

This helper saves both the ordinary compiled circuit diagram and the logical-inverse-prefix diagram to `demo_outputs/` as SVG and PNG files.


In [12]:
from pathlib import Path
import re
import subprocess


def extract_svg_text(diagram_obj) -> str:
    text = str(diagram_obj)
    match = re.search(r"(<svg\b.*</svg>)", text, flags=re.DOTALL)
    if match is None:
        raise RuntimeError("Could not find an <svg>...</svg> block in the diagram output.")
    return match.group(1)


def save_diagram_png(circuit, stem: str, *, out_dir: Path, height: int = 520) -> dict[str, str]:
    out_dir.mkdir(parents=True, exist_ok=True)
    svg_path = out_dir / f"{stem}.svg"
    png_path = out_dir / f"{stem}.png"

    svg_text = extract_svg_text(circuit.diagram(height=height))
    svg_path.write_text(svg_text)

    subprocess.run(
        [
            "inkscape",
            str(svg_path),
            "--export-type=png",
            f"--export-filename={png_path}",
        ],
        check=True,
    )

    return {"svg": str(svg_path), "png": str(png_path)}


SAVE_BASIS = VISUALIZE_BASIS
SAVE_DIR = Path("demo_outputs")

saved_compiled = save_diagram_png(
    compiled_tasks[SAVE_BASIS].tsim_circuit,
    f"compiled_noisy_suffix_{SAVE_BASIS.lower()}",
    out_dir=SAVE_DIR,
)

saved_logical_inverse = save_diagram_png(
    logical_inverse_prefix_tsim_circuits[SAVE_BASIS],
    f"logical_inverse_prefix_{SAVE_BASIS.lower()}",
    out_dir=SAVE_DIR,
)

{
    "compiled": saved_compiled,
    "logical_inverse_prefix": saved_logical_inverse,
}


{'compiled': {'svg': 'demo_outputs/compiled_noisy_suffix_x.svg',
  'png': 'demo_outputs/compiled_noisy_suffix_x.png'},
 'logical_inverse_prefix': {'svg': 'demo_outputs/logical_inverse_prefix_x.svg',
  'png': 'demo_outputs/logical_inverse_prefix_x.png'}}

## Takeaway

This notebook now contains two distinct constructions:

- `pipeline_faithful_tasks[...]`: the runnable Gemini task that uses the hand-inserted noiseless prefix plus the compiler-emitted noisy suffix
- `logical_inverse_prefix_tsim_circuits[...]`: a visualization circuit built by prepending the inverse of the compiled logical encoding-plus-`MSD` block to the ordinary compiled noisy suffix

The second object is the one that matches the baseline structure you described in this thread.
